# v4: Planner-Controller Deep Research Agent

v3 的结果说明 controller bootstrap 有效，但固定查询和只打开两个 top 文档会把模型带偏。v4 改成两阶段控制：先让模型只做查询规划，不回答问题；再由控制器执行多 query 检索、轻量重排、自动打开更多候选文档并收集证据，最后再让模型作答。

本 notebook 只定义方案和代码。本地不要执行、不要安装依赖、不要构建索引、不要启动或调用 vLLM。

## 1. 配置与导入

v4 复用 v3 的 `search_many` 工具集合，不额外修改检索器。

In [2]:
from pathlib import Path
import json
import re
import sys
from typing import Any, Dict, List, Optional

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from agent.dataset_utils import load_jsonl
from agent.eval import run_evaluation
from agent.tools import build_searcher, get_v3_research_tool_specs_and_registry, make_initial_search_queries
from agent.vllm_client import VLLMClient

VLLM_BASE_URL = "http://127.0.0.1:8000/v1"
MODEL_NAME = "qwen_auto"
API_KEY = "dummy"

BM25_INDEX_PATH = str(project_root / "indexes" / "browsecomp_plus_bm25.sqlite")
HARD50_PATH = str(project_root / "browsecomp_plus_hard50.jsonl")
SUBMISSION_PATH = str(project_root / "runs" / "v4_submission.jsonl")
EVAL_OUTPUT_PATH = str(project_root / "runs" / "v4_eval_results.jsonl")

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)
searcher = build_searcher(index_path=BM25_INDEX_PATH)
tool_specs, tool_registry = get_v3_research_tool_specs_and_registry(
    searcher=searcher,
    k=8,
    snippet_max_chars=1500,
    document_window_chars=2600,
)


## 2. Prompt

v4 把“查询规划”和“最终回答”分开，降低模型在没有证据时直接猜答案的概率。

In [3]:
PLANNER_SYSTEM_PROMPT = """
You generate search plans for BrowseComp-Plus BM25. Do not answer the question.
Return only compact JSON. Queries should be short, literal, and evidence-seeking.
Use names, dates, quoted phrases, unusual nouns, titles, institutions, and constraints.
""".strip()

ANSWER_SYSTEM_PROMPT = """
You are a Deep Research Agent for BrowseComp-Plus.
Use only local evidence already retrieved or returned by tools.
The controller has searched and opened candidate documents. Your task is to reason over that evidence.

Rules:
1. Do not answer from memory.
2. Prefer the answer that satisfies all clues, not merely the most familiar entity.
3. If evidence conflicts, explain the conflict briefly and choose the best-supported answer.
4. Do not output 'Unable to determine' if the evidence contains a plausible named entity, title, date, number, place, team, or organization that satisfies most clues.
5. The final answer must be short and appear after 'Exact Answer:'.

Final answer format:
Explanation: <brief evidence-based reasoning>
Evidence: <docid: support summary; repeat if needed>
Exact Answer: <short final answer only>
Confidence: <0-100>%
""".strip()

FINALIZER_PROMPT = """
Finalize now. Do not call more tools.
Use only evidence already present in the conversation and compressed state.
Return exactly:
Explanation: ...
Evidence: ...
Exact Answer: ...
Confidence: ...%
""".strip()


## 3. JSON、答案抽取与轻量重排

v4 的重排不替换 BM25，只是在 BM25 返回结果内部用问题关键词和 planner keywords 做轻量排序，以决定优先打开哪些文档。

In [4]:
STOPWORDS = {
    "the", "and", "for", "that", "with", "from", "this", "what", "which", "who", "whose", "where",
    "when", "about", "into", "between", "after", "before", "during", "their", "there", "would",
    "could", "should", "please", "provide", "identify", "looking", "question", "answer", "according",
}


def canonical_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "").strip().lower())


def truncate_text(text: str, max_chars: int) -> str:
    text = str(text or "")
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "... [truncated]"


def strip_thinking(text: str) -> str:
    text = str(text or "")
    text = re.sub(r"(?is)<think>.*?</think>\s*", "", text).strip()
    text = re.sub(r"(?is)<think>.*", "", text).strip()
    return text


def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    text = strip_thinking(text)
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        value = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None


def normalize_string_list(value: Any, limit: int = 10, max_chars: int = 180) -> List[str]:
    if isinstance(value, str):
        items = [value]
    elif isinstance(value, list):
        items = value
    else:
        items = []
    output = []
    seen = set()
    for item in items:
        text = " ".join(str(item or "").replace("\n", " ").split()).strip(" ,.;:")
        if len(text) > max_chars:
            text = text[:max_chars].rstrip()
        key = text.lower()
        if text and key not in seen:
            output.append(text)
            seen.add(key)
        if len(output) >= limit:
            break
    return output


def tokenize_for_score(text: str) -> List[str]:
    tokens = re.findall(r"[A-Za-z0-9][A-Za-z0-9_\-']+", str(text or "").lower())
    return [token.strip("'-") for token in tokens if len(token.strip("'-")) >= 4 and token not in STOPWORDS]


def build_scoring_terms(question: str, planner: Dict[str, Any]) -> List[str]:
    terms = tokenize_for_score(question)
    for key in ("key_phrases", "keywords", "entities", "answer_type"):
        terms.extend(tokenize_for_score(" ".join(normalize_string_list(planner.get(key), limit=20))))
    deduped = []
    seen = set()
    for term in terms:
        if term not in seen:
            deduped.append(term)
            seen.add(term)
    return deduped[:80]


def rank_search_results(results: List[Dict[str, Any]], scoring_terms: List[str], limit: int = 8) -> List[Dict[str, Any]]:
    ranked = []
    for item in results:
        haystack = canonical_text(" ".join([item.get("query", ""), item.get("snippet", ""), item.get("url", "")]))
        overlap = sum(1 for term in scoring_terms if term in haystack)
        score = float(item.get("score", 0) or 0)
        ranked.append({**item, "controller_score": overlap * 1000 + score, "term_overlap": overlap})
    ranked.sort(key=lambda row: (row.get("term_overlap", 0), row.get("score", 0)), reverse=True)
    return ranked[:limit]


def extract_exact_answer(text: str) -> str:
    original = str(text or "").strip()
    clean = strip_thinking(original)
    for candidate_text in (clean, original):
        match = re.search(r"(?im)^\s*Exact Answer\s*:\s*(.+?)\s*$", candidate_text)
        if match:
            return match.group(1).strip()
        match = re.search(r"(?im)^\s*Answer\s*:\s*(.+?)\s*$", candidate_text)
        if match:
            return match.group(1).strip()
    if clean and len(clean) <= 160:
        return clean
    return "Unable to determine"


def compact_result_summary(results: List[Dict[str, Any]], max_items: int = 10) -> List[Dict[str, Any]]:
    compact = []
    for item in results[:max_items]:
        compact.append(
            {
                "docid": item.get("docid", ""),
                "score": item.get("score", 0),
                "controller_score": item.get("controller_score", 0),
                "url": item.get("url", ""),
                "query": item.get("query", ""),
                "snippet": truncate_text(item.get("snippet", ""), 500),
            }
        )
    return compact


## 4. 工具执行与状态管理

控制器执行的搜索、打开窗口和证据收集仍写入 `messages`，因此提交轨迹可以被 `agent/eval.py` 还原和统计。

In [5]:
def parse_tool_arguments(raw_arguments: Any) -> Dict[str, Any]:
    if isinstance(raw_arguments, dict):
        return raw_arguments
    if raw_arguments in (None, ""):
        return {}
    try:
        parsed = json.loads(raw_arguments)
    except json.JSONDecodeError:
        return {"_argument_parse_error": str(raw_arguments)}
    return parsed if isinstance(parsed, dict) else {"_argument_parse_error": str(raw_arguments)}


def normalize_tool_result(tool_name: str, result: Any, max_document_chars: int = 4500) -> Any:
    if tool_name == "get_document" and isinstance(result, dict):
        normalized = dict(result)
        if "text" in normalized:
            full_text = str(normalized.get("text", ""))
            normalized["text"] = truncate_text(full_text, max_document_chars)
            normalized["text_length"] = len(full_text)
            normalized["text_is_truncated"] = len(full_text) > max_document_chars
        return normalized
    return result


def execute_tool(tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    if tool_name not in tool_registry:
        return {"ok": False, "tool_name": tool_name, "arguments": arguments, "error": "unknown tool"}
    try:
        raw_result = tool_registry[tool_name](**arguments)
        return {
            "ok": True,
            "tool_name": tool_name,
            "arguments": arguments,
            "tool_result": normalize_tool_result(tool_name, raw_result),
        }
    except Exception as exc:
        return {"ok": False, "tool_name": tool_name, "arguments": arguments, "error": repr(exc)}


def execute_tool_call(tool_call: Dict[str, Any]) -> Dict[str, Any]:
    function = tool_call.get("function", {}) or {}
    tool_name = function.get("name", "")
    arguments = parse_tool_arguments(function.get("arguments", "{}"))
    if "_argument_parse_error" in arguments:
        return {"ok": False, "tool_name": tool_name, "arguments": arguments, "error": "invalid JSON arguments"}
    return execute_tool(tool_name, arguments)


def make_tool_message(tool_call_id: str, executed: Dict[str, Any]) -> Dict[str, Any]:
    content = executed.get("tool_result") if executed.get("ok") else executed
    return {"role": "tool", "tool_call_id": tool_call_id, "content": json.dumps(content, ensure_ascii=False)}


def docids_from_tool_result(tool_name: str, result: Any) -> List[str]:
    if tool_name in {"search", "search_many"} and isinstance(result, list):
        return [str(item.get("docid", "")) for item in result if isinstance(item, dict) and item.get("docid")]
    if tool_name == "collect_evidence_snippets" and isinstance(result, dict):
        return [str(item.get("docid", "")) for item in result.get("snippets", []) if item.get("docid")]
    if isinstance(result, dict) and result.get("docid"):
        return [str(result.get("docid"))]
    return []


def initial_state(question: str) -> Dict[str, Any]:
    return {
        "question": question,
        "planner": {},
        "search_queries": [],
        "seen_docids": [],
        "opened_docids": [],
        "ranked_results": [],
        "evidence_table": [],
        "tool_events": [],
        "rounds": [],
        "low_gain_rounds": 0,
        "controller_actions": [],
    }


def summarize_tool_result(tool_name: str, result: Any) -> str:
    if tool_name in {"search", "search_many"}:
        return "docids=" + ", ".join(docids_from_tool_result(tool_name, result)[:12])
    if tool_name == "collect_evidence_snippets" and isinstance(result, dict):
        return f"collected {len(result.get('snippets', []))} snippets"
    if tool_name == "find_in_document" and isinstance(result, dict):
        return f"{result.get('num_matches', 0)} matches for {result.get('keyword', '')} in {result.get('docid', '')}"
    if isinstance(result, dict) and result.get("docid"):
        return f"inspected docid={result.get('docid', '')}"
    return truncate_text(json.dumps(result, ensure_ascii=False), 300)


def update_state_from_execution(state: Dict[str, Any], executed: Dict[str, Any], round_id: int) -> int:
    tool_name = executed.get("tool_name", "")
    arguments = executed.get("arguments", {}) or {}
    result = executed.get("tool_result")
    new_signal = 0

    if tool_name == "search_many":
        for query in arguments.get("queries", []) or []:
            if query and query not in state["search_queries"]:
                state["search_queries"].append(query)
                new_signal += 1
    if tool_name == "search":
        query = arguments.get("query", "")
        if query and query not in state["search_queries"]:
            state["search_queries"].append(query)
            new_signal += 1

    for docid in docids_from_tool_result(tool_name, result):
        if docid and docid not in state["seen_docids"]:
            state["seen_docids"].append(docid)
            new_signal += 1
        if tool_name in {"get_document", "get_document_window", "find_in_document"} and docid not in state["opened_docids"]:
            state["opened_docids"].append(docid)
            new_signal += 1

    if tool_name in {"find_in_document", "collect_evidence_snippets"}:
        state["evidence_table"].append(
            {"round_id": round_id, "tool_name": tool_name, "arguments": arguments, "summary": summarize_tool_result(tool_name, result)}
        )

    state["tool_events"].append(
        {"round_id": round_id, "tool_name": tool_name, "arguments": arguments, "ok": executed.get("ok", False), "summary": summarize_tool_result(tool_name, result)}
    )
    return new_signal


def append_controller_tool_call(messages: List[Dict[str, Any]], state: Dict[str, Any], tool_name: str, arguments: Dict[str, Any], round_id: int, note: str) -> Dict[str, Any]:
    call_id = f"controller_{round_id}_{len(state['tool_events']) + 1}_{tool_name}"
    messages.append(
        {
            "role": "assistant",
            "content": note,
            "tool_calls": [
                {"id": call_id, "type": "function", "function": {"name": tool_name, "arguments": json.dumps(arguments, ensure_ascii=False)}}
            ],
        }
    )
    executed = execute_tool(tool_name, arguments)
    messages.append(make_tool_message(call_id, executed))
    update_state_from_execution(state, executed, round_id)
    state["controller_actions"].append({"round_id": round_id, "tool_name": tool_name, "note": note})
    return executed


def public_state_summary(state: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "planner": state["planner"],
        "search_queries": state["search_queries"][-12:],
        "seen_docids": state["seen_docids"][-40:],
        "opened_docids": state["opened_docids"][-20:],
        "ranked_results": compact_result_summary(state["ranked_results"], max_items=8),
        "evidence_table": state["evidence_table"][-12:],
        "recent_tool_events": state["tool_events"][-14:],
    }


## 5. Planner-Controller Bootstrap

v4 的关键变化：模型先生成 search plan，控制器执行并重排，再自动打开 top candidates 和收集关键词证据。

In [6]:
def build_planner_fallback(question: str) -> Dict[str, Any]:
    return {
        "search_queries": make_initial_search_queries(question, max_queries=6, max_query_chars=180),
        "key_phrases": [],
        "keywords": tokenize_for_score(question)[:20],
        "answer_type": "short answer",
    }


def plan_search(question: str, max_tokens: int = 900) -> Dict[str, Any]:
    fallback = build_planner_fallback(question)
    user_prompt = {
        "role": "user",
        "content": (
            "Question:\n"
            f"{question}\n\n"
            "Return JSON with keys:\n"
            "search_queries: 4-8 concise BM25 queries, each under 16 words;\n"
            "key_phrases: important literal phrases or names;\n"
            "keywords: clue words to find in documents;\n"
            "answer_type: expected answer type such as person, title, date, organization, place, number.\n"
            "Do not answer the question."
        ),
    }
    try:
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": PLANNER_SYSTEM_PROMPT}, user_prompt],
            temperature=0.0,
            max_tokens=max_tokens,
        )
        text = response["choices"][0]["message"].get("content", "")
        parsed = extract_json_object(text)
    except Exception:
        parsed = None
    if not parsed:
        parsed = fallback

    deterministic = make_initial_search_queries(question, max_queries=5, max_query_chars=180)
    planner_queries = normalize_string_list(parsed.get("search_queries"), limit=8, max_chars=180)
    key_phrases = normalize_string_list(parsed.get("key_phrases"), limit=12, max_chars=120)
    keywords = normalize_string_list(parsed.get("keywords"), limit=16, max_chars=60)
    all_queries = normalize_string_list(deterministic + planner_queries + key_phrases, limit=10, max_chars=180)
    return {
        "search_queries": all_queries,
        "planner_queries": planner_queries,
        "key_phrases": key_phrases,
        "keywords": keywords,
        "answer_type": str(parsed.get("answer_type", fallback.get("answer_type", "short answer"))),
    }


def bootstrap_research_v4(
    question: str,
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    per_query_k: int = 6,
    auto_open_docs: int = 4,
) -> None:
    planner = plan_search(question)
    state["planner"] = planner
    messages.append({"role": "assistant", "content": "Controller planner:\n" + json.dumps(planner, ensure_ascii=False, indent=2)})

    executed = append_controller_tool_call(
        messages=messages,
        state=state,
        tool_name="search_many",
        arguments={"queries": planner["search_queries"], "per_query_k": per_query_k},
        round_id=0,
        note="Controller bootstrap: execute planned BM25 searches.",
    )
    raw_results = executed.get("tool_result", []) if executed.get("ok") else []
    scoring_terms = build_scoring_terms(question, planner)
    ranked = rank_search_results(raw_results, scoring_terms=scoring_terms, limit=10)
    state["ranked_results"] = ranked
    messages.append({"role": "assistant", "content": "Controller ranked candidates:\n" + json.dumps(compact_result_summary(ranked), ensure_ascii=False, indent=2)})

    opened = 0
    for item in ranked:
        docid = str(item.get("docid", ""))
        if not docid or docid in state["opened_docids"]:
            continue
        append_controller_tool_call(
            messages=messages,
            state=state,
            tool_name="get_document_window",
            arguments={"docid": docid, "start": 0, "window_chars": 2600},
            round_id=0,
            note="Controller bootstrap: inspect ranked candidate document.",
        )
        opened += 1
        if opened >= auto_open_docs:
            break

    keywords = ", ".join(normalize_string_list(planner.get("keywords") + planner.get("key_phrases"), limit=12, max_chars=80))
    top_docids = [str(item.get("docid", "")) for item in ranked[:8] if item.get("docid")]
    if keywords and top_docids:
        append_controller_tool_call(
            messages=messages,
            state=state,
            tool_name="collect_evidence_snippets",
            arguments={"docids": top_docids, "keywords": keywords, "window_chars": 900, "max_snippets": 12},
            round_id=0,
            note="Controller bootstrap: collect keyword evidence snippets from ranked candidates.",
        )


## 6. Agent Loop

bootstrap 后仍允许模型继续调用工具，但 v4 的主要证据由控制器先准备好，避免 v3 中只读两个错误文档就收束。

In [7]:
def build_model_messages(messages: List[Dict[str, Any]], state: Dict[str, Any], recent_message_limit: int = 24) -> List[Dict[str, Any]]:
    state_message = {
        "role": "user",
        "content": "Compressed research state:\n" + json.dumps(public_state_summary(state), ensure_ascii=False, indent=2),
    }
    recent = messages[2:]
    if len(recent) > recent_message_limit:
        recent = recent[-recent_message_limit:]
        while recent and recent[0].get("role") == "tool":
            recent = recent[1:]
    return [messages[0], messages[1], state_message] + recent


def response_to_assistant_message(response: Dict[str, Any]) -> tuple[Dict[str, Any], List[Dict[str, Any]]]:
    message = response["choices"][0]["message"]
    assistant_message = {"role": "assistant", "content": message.get("content") or ""}
    tool_calls = message.get("tool_calls") or []
    if tool_calls:
        assistant_message["tool_calls"] = tool_calls
    return assistant_message, tool_calls


def finalize_answer(messages: List[Dict[str, Any]], state: Dict[str, Any], reason: str, max_tokens: int = 1200) -> str:
    final_instruction = {"role": "user", "content": reason + "\n\n" + FINALIZER_PROMPT}
    response = client.simple_chat(
        model=MODEL_NAME,
        messages=build_model_messages(messages, state) + [final_instruction],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    assistant_message, _ = response_to_assistant_message(response)
    messages.append(final_instruction)
    messages.append({"role": "assistant", "content": assistant_message.get("content", "")})
    return assistant_message.get("content", "")


def run_v4_agent(
    question: str,
    query_id: Optional[str] = None,
    max_rounds: int = 4,
    low_gain_round_limit: int = 2,
    auto_open_docs: int = 4,
    per_query_k: int = 6,
    max_tokens: int = 1400,
    recent_message_limit: int = 24,
) -> Dict[str, Any]:
    messages: List[Dict[str, Any]] = [
        {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    state = initial_state(question)
    bootstrap_research_v4(question, messages, state, per_query_k=per_query_k, auto_open_docs=auto_open_docs)

    final_output = ""
    status = "max_rounds_reached"

    for round_id in range(1, max_rounds + 1):
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=build_model_messages(messages, state, recent_message_limit=recent_message_limit),
            temperature=0.0,
            max_tokens=max_tokens,
            tools=tool_specs,
            tool_choice="auto",
        )
        assistant_message, tool_calls = response_to_assistant_message(response)
        messages.append(assistant_message)
        state["rounds"].append(
            {"round_id": round_id, "assistant_content": truncate_text(assistant_message.get("content", ""), 500), "tool_call_count": len(tool_calls)}
        )

        if not tool_calls:
            status = "completed_after_planner_controller"
            final_output = finalize_answer(messages, state, reason="Use the planner-controller evidence to produce the final answer.")
            break

        round_signal = 0
        for tool_call in tool_calls:
            executed = execute_tool_call(tool_call)
            messages.append(make_tool_message(tool_call.get("id", ""), executed))
            round_signal += update_state_from_execution(state, executed, round_id)

        if round_signal == 0:
            state["low_gain_rounds"] += 1
        else:
            state["low_gain_rounds"] = 0
        if state["low_gain_rounds"] >= low_gain_round_limit:
            status = "stopped_low_gain_search"
            final_output = finalize_answer(messages, state, reason="Recent tool calls added no new useful evidence.")
            break
    else:
        final_output = finalize_answer(messages, state, reason=f"Reached max_rounds={max_rounds}.")

    return {
        "query_id": query_id,
        "status": status,
        "predicted_answer": extract_exact_answer(final_output),
        "final_output": final_output,
        "messages": messages,
        "state_summary": public_state_summary(state),
    }


## 7. Submission 与评测

评测仍使用课程提供的 `run_evaluation()`，保持 `max_tokens=1024`，避免 reasoning evaluator 被截断。

In [8]:
def build_submission_record(row: Dict[str, Any], **agent_kwargs: Any) -> Dict[str, Any]:
    result = run_v4_agent(question=row["query"], query_id=str(row.get("query_id", "")), **agent_kwargs)
    return {
        "query_id": str(row.get("query_id", "")),
        "predicted_answer": result["predicted_answer"],
        "status": result["status"],
        "messages": result["messages"],
        "state_summary": result["state_summary"],
    }


def generate_submission(
    dataset_path: str = HARD50_PATH,
    output_path: str = SUBMISSION_PATH,
    limit: Optional[int] = 50,
    **agent_kwargs: Any,
) -> List[Dict[str, Any]]:
    rows = load_jsonl(dataset_path, limit=limit)
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    records = []
    with output_file.open("w", encoding="utf-8") as fout:
        for index, row in enumerate(rows, start=1):
            print(f"[{index}/{len(rows)}] query_id={row.get('query_id', '')}")
            record = build_submission_record(row, **agent_kwargs)
            records.append(record)
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} records to {output_path}")
    return records


def evaluate_submission(
    submission_path: str = SUBMISSION_PATH,
    dataset_path: str = HARD50_PATH,
    output_path: str = EVAL_OUTPUT_PATH,
):
    return run_evaluation(
        submission_path=submission_path,
        dataset_path=dataset_path,
        model_name=MODEL_NAME,
        base_url=VLLM_BASE_URL,
        api_key=API_KEY,
        output_path=output_path,
        max_tokens=1024,
        verbose=True,
    )


## 8. 服务器运行入口

本地不要执行。到服务器确认 vLLM 和 BM25 索引已就绪后，再取消注释运行。

```python
# records = generate_submission(limit=50, max_rounds=4, low_gain_round_limit=2)
# summary, details = evaluate_submission()
```

In [9]:
records = generate_submission(limit=50, max_rounds=4, low_gain_round_limit=2)
summary, details = evaluate_submission()

[1/50] query_id=442
[2/50] query_id=549
[3/50] query_id=26
[4/50] query_id=471
[5/50] query_id=761
[6/50] query_id=815
[7/50] query_id=216
[8/50] query_id=314
[9/50] query_id=241
[10/50] query_id=930
[11/50] query_id=651
[12/50] query_id=1082
[13/50] query_id=159
[14/50] query_id=5
[15/50] query_id=395
[16/50] query_id=944
[17/50] query_id=747
[18/50] query_id=635
[19/50] query_id=113
[20/50] query_id=556
[21/50] query_id=153
[22/50] query_id=263
[23/50] query_id=827
[24/50] query_id=730
[25/50] query_id=79
[26/50] query_id=86
[27/50] query_id=397
[28/50] query_id=427
[29/50] query_id=1072
[30/50] query_id=380
[31/50] query_id=662
[32/50] query_id=36
[33/50] query_id=53
[34/50] query_id=709
[35/50] query_id=166
[36/50] query_id=869
[37/50] query_id=432
[38/50] query_id=297
[39/50] query_id=371
[40/50] query_id=785
[41/50] query_id=978
[42/50] query_id=450
[43/50] query_id=342
[44/50] query_id=1061
[45/50] query_id=1095
[46/50] query_id=33
[47/50] query_id=487
[48/50] query_id=180
[49/5